In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.44.0 datasets==2.21.0 accelerate==0.34.0
!pip install peft==0.12.0 bitsandbytes==0.43.0 trl==0.10.0
!pip install autoawq==0.2.6 vllm==0.6.0
!pip install fastapi==0.115.0 uvicorn[standard]==0.30.0 slowapi==0.1.9
!pip install httpx==0.27.0 python-dotenv==1.0.0 pyyaml==6.0.0
!pip install nltk==3.9.0 rouge-score==0.1.2 evaluate==0.4.0
!pip install matplotlib==3.9.0 seaborn==0.13.0 pandas==2.2.0
!pip install wandb tensorboard

# Stage 4 — Capstone Project
## Notebook 3: End-to-End LLM Fine-Tuning & Deployment Pipeline

**What you will build:** A complete production pipeline taking a base LLM from raw data through fine-tuning, alignment, quantization, deployment, and evaluation.

**9 Parts:**
1. Dataset creation with domain-specific templates
2. QLoRA supervised fine-tuning (SFT)
3. DPO preference alignment
4. AWQ model quantization
5. vLLM inference deployment
6. FastAPI gateway with auth + rate limiting
7. Docker containerization
8. Automated evaluation (base vs fine-tuned)
9. Production deployment documentation

**Target hardware:** NVIDIA RTX 3090 / 4090 (24 GB VRAM), 64 GB RAM, 100+ GB storage.

Every section includes `<YOUR_...>` markers for domain-specific customization.

---
## Project Directory Structure

```
capstone-project/
├── data/
│   ├── raw/                  # Raw documents
│   ├── processed/            # Cleaned data
│   ├── instructions.jsonl    # Instruction dataset
│   └── preferences.jsonl     # DPO preference pairs
├── models/
│   ├── base/                 # Base model
│   ├── qlora-adapter/        # QLoRA adapter weights
│   ├── dpo-checkpoint/       # DPO model
│   └── awq-quantized/        # AWQ 4-bit model
├── src/
│   ├── dataset.py, train_qlora.py, train_dpo.py
│   ├── quantize_awq.py, evaluate.py, utils.py
├── gateway/
│   ├── main.py, auth.py, rate_limit.py
│   └── requirements.txt, .env.example
├── docker/
│   ├── Dockerfile.gateway, Dockerfile.vllm
│   └── docker-compose.yml (at root)
├── docs/
│   ├── DEPLOYMENT.md, API.md, TROUBLESHOOTING.md
├── configs/
│   ├── qlora_config.yaml, vllm_config.yaml
├── requirements.txt
└── README.md
```

In [ ]:
# Global setup: imports, paths, reproducibility
import os, json, yaml, random, time
from pathlib import Path
from typing import List, Dict
import torch, numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from trl import SFTTrainer, DPOTrainer, DPOConfig

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Paths — CHANGE THIS
PROJECT_ROOT = Path("<YOUR_PROJECT_ROOT>")
DATA_DIR, MODEL_DIR = PROJECT_ROOT / "data", PROJECT_ROOT / "models"

# Create directory structure
for d in ["data/raw", "data/processed", "models/base", "models/qlora-adapter",
          "models/dpo-checkpoint", "models/awq-quantized", "src", "gateway", "docker",
          "docs", "configs"]:
    Path(PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Part 1: Dataset Creation with Domain-Specific Examples

Template-based generation of instruction-tuning data. Replace all `<YOUR_...>` placeholders with domain-specific values.

**Format (Alpaca):** `{"instruction": "...", "input": "...", "output": "..."}`

In [ ]:
# ---- Part 1a: Templates & Seed Data ----
# Replace with YOUR domain's templates and vocabulary
INSTRUCTION_TEMPLATES = [
    "Explain {concept} in simple terms.",
    "What are the potential side effects of {item}?",
    "A patient presents with {symptoms}. What is the differential diagnosis?",
    "Describe the standard treatment protocol for {condition}.",
    "Compare {item_a} and {item_b} for treating {condition}.",
    "What tests should be ordered for {symptoms}?",
    "Summarize the latest guidelines for managing {condition}.",
    "What are the contraindications for {item}?",
    "Explain the mechanism of {concept} step by step.",
    "A user asks: '{question}'. Draft an accurate response.",
]

SEED_DATA = {
    "concept": ["<YOUR_CONCEPT_1>", "<YOUR_CONCEPT_2>", "<YOUR_CONCEPT_3>"],
    "item": ["<YOUR_ITEM_1>", "<YOUR_ITEM_2>", "<YOUR_ITEM_3>"],
    "symptoms": ["<YOUR_SYMPTOM_SET_1>", "<YOUR_SYMPTOM_SET_2>"],
    "condition": ["<YOUR_CONDITION_1>", "<YOUR_CONDITION_2>"],
    "item_a": ["<YOUR_ITEM_A1>", "<YOUR_ITEM_A2>"],
    "item_b": ["<YOUR_ITEM_B1>", "<YOUR_ITEM_B2>"],
    "question": ["<YOUR_FAQ_1>", "<YOUR_FAQ_2>"],
}

print(f"{len(INSTRUCTION_TEMPLATES)} templates, "
      f"{sum(len(v) for v in SEED_DATA.values())} seed values")

In [ ]:
# ---- Part 1b: Generate, Validate, and Save Dataset ----
def generate_instruction(idx):
    """Generate one example by filling template placeholders."""
    template = INSTRUCTION_TEMPLATES[idx % len(INSTRUCTION_TEMPLATES)]
    instruction = template
    for key, values in SEED_DATA.items():
        placeholder = "{" + key + "}"
        if placeholder in instruction:
            instruction = instruction.replace(placeholder, random.choice(values), 1)
    return {"instruction": instruction, "input": "",
            "output": f"[Reference answer for: {instruction[:80]}...]"}

def generate_dataset(n=100, seed=42):
    random.seed(seed)
    return [generate_instruction(i) for i in range(n)]

def validate_dataset(ds):
    report = {"total": len(ds), "errors": [], "warnings": []}
    for i, ex in enumerate(ds):
        if not ex["instruction"].strip(): report["errors"].append(f"{i}: empty instruction")
        if not ex["output"].strip(): report["errors"].append(f"{i}: empty output")
        if "{" in ex["instruction"]: report["errors"].append(f"{i}: unfilled placeholder")
    report["valid"] = len(report["errors"]) == 0
    return report

# ChatML format — customize for your model
def format_chatml(ex):
    user_msg = ex["instruction"]
    if ex.get("input"): user_msg += f"\n\nContext: {ex['input']}"
    return (f"<|im_start|>system\n<YOUR_SYSTEM_PROMPT>\n<|im_end|>\n"
            f"<|im_start|>user\n{user_msg}\n<|im_end|>\n"
            f"<|im_start|>assistant\n{ex['output']}\n<|im_end|>")

# Generate & save
raw = generate_dataset(n=100)
hf_ds = Dataset.from_list(raw)
split = hf_ds.train_test_split(test_size=0.2, seed=SEED)
train_ds, val_ds = split["train"], split["test"]

hf_ds.to_json(str(DATA_DIR / "instructions.jsonl"))
train_ds.to_json(str(DATA_DIR / "train.jsonl"))
val_ds.to_json(str(DATA_DIR / "val.jsonl"))

print(json.dumps(validate_dataset(hf_ds), indent=2))
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"\nSample formatted:\n{format_chatml(raw[0])[:250]}...")

---
## Part 2: QLoRA SFT Training with Full Config

QLoRA = 4-bit NF4 base model + LoRA adapters. VRAM: ~8-12 GB for a 7-8B model.

In [ ]:
# ---- Part 2: QLoRA Config & SFT Trainer ----

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

# LoRA adapter
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)

# Training args — tune these for your setup
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "qlora-adapter"),
    num_train_epochs=<YOUR_NUM_EPOCHS>,            # Typically 1-3
    per_device_train_batch_size=<YOUR_BATCH_SIZE>,  # 1-4 depending on VRAM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=<YOUR_GRAD_ACCUM>,  # effective = bs * accum * gpus
    gradient_checkpointing=True,
    optim="paged_adamw_8bit", learning_rate=2e-4,
    lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.001, max_grad_norm=0.3,
    logging_steps=10, eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=100, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    bf16=True, tf32=True, seed=SEED, report_to="wandb",
    run_name="<YOUR_WANDB_RUN>", remove_unused_columns=False)

BASE_MODEL_ID = "<YOUR_BASE_MODEL>"  # e.g., meta-llama/Meta-Llama-3-8B

# Load model (uncomment to train)
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config,
#     device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16)
# model = prepare_model_for_kbit_training(model)
# model.gradient_checkpointing_enable()
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()  # Expected: ~0.5% trainable
#
# trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_ds,
#     eval_dataset=val_ds, tokenizer=tokenizer, formatting_func=format_chatml, max_seq_length=2048)
# trainer.train()
# trainer.save_model()

# Merge & save helper
def merge_and_save(base_id, adapter_path, out_path):
    base = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.bfloat16, device_map="auto")
    merged = PeftModel.from_pretrained(base, adapter_path).merge_and_unload()
    merged.save_pretrained(out_path, safe_serialization=True)
    AutoTokenizer.from_pretrained(base_id).save_pretrained(out_path)
    print(f"[OK] Merged model saved to {out_path}")

print("QLoRA config and training skeleton ready.")
print(f"LoRA: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"LR: {training_args.learning_rate}, Epochs: {training_args.num_train_epochs}")

---
## Part 3: DPO Preference Data Creation & DPOTrainer Setup

DPO aligns the model with human preferences using chosen/rejected pairs -- no separate reward model needed.

**Data format:** `{"prompt": "...", "chosen": "good response", "rejected": "bad response"}`

In [ ]:
# ---- Part 3: DPO Preference Data & Training ----
def create_preference_pair(instruction, good_response, bad_response):
    prompt = (f"<|im_start|>system\n<YOUR_SYSTEM_PROMPT>\n<|im_end|>\n"
              f"<|im_start|>user\n{instruction}\n<|im_end|>\n<|im_start|>assistant\n")
    return {"prompt": prompt, "chosen": good_response, "rejected": bad_response}

# Build preference pairs — replace with domain-specific examples
PREFERENCE_PAIRS_DATA = [
    {"instruction": "<YOUR_INSTRUCTION_1>",
     "chosen": "<YOUR_GOOD_RESPONSE_1 — accurate, comprehensive, helpful>",
     "rejected": "<YOUR_BAD_RESPONSE_1 — incomplete, misleading, or unhelpful>"},
    {"instruction": "<YOUR_INSTRUCTION_2>",
     "chosen": "<YOUR_GOOD_RESPONSE_2>",
     "rejected": "<YOUR_BAD_RESPONSE_2>"},
    {"instruction": "<YOUR_INSTRUCTION_3>",
     "chosen": "<YOUR_GOOD_RESPONSE_3>",
     "rejected": "<YOUR_BAD_RESPONSE_3>"},
    # Add 10-20 pairs minimum for meaningful DPO
]

pref_pairs = [create_preference_pair(it["instruction"], it["chosen"], it["rejected"])
              for it in PREFERENCE_PAIRS_DATA]
pref_ds = Dataset.from_list(pref_pairs)
pref_ds.to_json(str(DATA_DIR / "preferences.jsonl"))
print(f"Created {len(pref_ds)} preference pairs -> {DATA_DIR / 'preferences.jsonl'}")

# DPO config
dpo_config = DPOConfig(
    output_dir=str(MODEL_DIR / "dpo-checkpoint"),
    num_train_epochs=1, per_device_train_batch_size=2, gradient_accumulation_steps=4,
    beta=0.1, max_prompt_length=512, max_length=1024,
    learning_rate=5e-5, lr_scheduler_type="cosine", warmup_ratio=0.1,
    optim="adamw_8bit", logging_steps=10, report_to="wandb",
    eval_strategy="steps", eval_steps=50, save_steps=100, save_total_limit=2,
    bf16=True, seed=SEED, remove_unused_columns=False,
    run_name="<YOUR_WANDB_DPO_RUN>")

print(f"DPO config: beta={dpo_config.beta}, LR={dpo_config.learning_rate}")
print("To train: dpo_trainer = DPOTrainer(model=..., args=dpo_config, ...); dpo_trainer.train()")

---
## Part 4: AWQ Quantization of Fine-Tuned Model

AWQ (Activation-Aware Weight Quantization) compresses to 4-bit with ~3.5x size reduction by protecting salient weight channels.

In [ ]:
# ---- Part 4: AWQ Quantization & Size Comparison ----
MODEL_TO_QUANTIZE = "<YOUR_DPO_MODEL_PATH>"
AWQ_OUTPUT_DIR = str(MODEL_DIR / "awq-quantized")

awq_quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}

AWQ_SCRIPT = '''
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model = AutoAWQForCausalLM.from_pretrained("<YOUR_MODEL_PATH>", safetensors=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("<YOUR_MODEL_PATH>", trust_remote_code=True)

model.quantize(tokenizer, quant_config={"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"},
               calib_data="pileval")  # or custom calibration data
model.save_quantized("<YOUR_OUTPUT_DIR>", safetensors=True)
tokenizer.save_pretrained("<YOUR_OUTPUT_DIR>")
print("[OK] AWQ quantization complete.")
'''

with open(PROJECT_ROOT / "src" / "quantize_awq.py", "w") as f:
    f.write(AWQ_SCRIPT)

# Size comparison visualization
model_sizes = {"Base (FP16)": 14.8, "QLoRA Adapter": 0.16,
                "SFT+DPO (FP16)": 14.9, "AWQ INT4": 4.2}

fig, ax = plt.subplots(figsize=(10, 4.5))
names, sizes = list(model_sizes.keys()), list(model_sizes.values())
colors = ["#607D8B", "#4CAF50", "#2196F3", "#FF9800"]
bars = ax.barh(names, sizes, color=colors, edgecolor="white", linewidth=1.5)
for bar, s in zip(bars, sizes):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{s:.1f} GB", va="center", fontsize=11, fontweight="bold")
ax.set_xlabel("Size (GB)"); ax.set_title("Model Size Comparison Across Pipeline Stages")
ax.set_xlim(0, max(sizes) * 1.35); ax.grid(axis="x", alpha=0.3)
compression = model_sizes["SFT+DPO (FP16)"] / model_sizes["AWQ INT4"]
ax.annotate(f"~{compression:.1f}x compression", xy=(model_sizes["AWQ INT4"], 2),
            xytext=(max(sizes) * 0.5, 2.3), arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
            fontsize=12, fontweight="bold", color="#D32F2F")
plt.tight_layout(); plt.show()
print(f"\nAWQ script saved -> src/quantize_awq.py")

---
## Part 5: vLLM Deployment Configuration

vLLM with PagedAttention, continuous batching, and native AWQ support.

In [ ]:
# ---- Part 5: vLLM Config & Launch ----

vllm_config = {
    "model": "<YOUR_AWQ_MODEL_PATH>", "host": "0.0.0.0", "port": 8000,
    "served_model_name": "<YOUR_MODEL_NAME>",
    "max_model_len": 4096, "max_num_seqs": 32,
    "gpu_memory_utilization": 0.90, "quantization": "awq",
    "dtype": "auto", "trust_remote_code": True,
}

with open(PROJECT_ROOT / "configs" / "vllm_config.yaml", "w") as f:
    yaml.dump(vllm_config, f, default_flow_style=False)

launch_cmd = f"""python -m vllm.entrypoints.openai.api_server \\
    --model {vllm_config['model']} --host {vllm_config['host']} --port {vllm_config['port']} \\
    --quantization {vllm_config['quantization']} --max-model-len {vllm_config['max_model_len']} \\
    --gpu-memory-utilization {vllm_config['gpu_memory_utilization']} \\
    --served-model-name {vllm_config['served_model_name']}"""

print("vLLM config saved.")
print(f"\nLaunch command:\n{launch_cmd}")

print("\nOpenAI-compatible client (when server is running):")
print('  curl http://localhost:8000/v1/models')
print('  curl http://localhost:8000/v1/chat/completions -H "Content-Type: application/json" \\')
print('    -d \'{"model": "<YOUR_MODEL_NAME>", "messages": [{"role": "user", "content": "Hello"}]}\'')

---
## Part 6: FastAPI Gateway with Auth & Rate Limiting

Production API gateway sitting in front of vLLM with API-key authentication, rate limiting, health checks, and OpenAPI docs at `/docs`.

In [ ]:
# ---- Part 6: FastAPI Gateway ----

GATEWAY_CODE = r'''
"""LLM API Gateway — FastAPI with auth, rate limiting, and vLLM routing."""
import os, time, hashlib, logging
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from slowapi import Limiter
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import httpx

load_dotenv()

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
API_KEYS = set(os.getenv("API_KEYS", "<YOUR_API_KEY_1>,<YOUR_API_KEY_2>").split(","))
REQUEST_TIMEOUT = int(os.getenv("REQUEST_TIMEOUT", "120"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="<YOUR_APP_NAME> LLM Gateway", version="1.0.0", docs_url="/docs")
limiter = Limiter(key_func=get_remote_address)
app.state.limiter = limiter
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])
security = HTTPBearer()

def verify_api_key(creds: HTTPAuthorizationCredentials = Depends(security)):
    if creds.credentials not in API_KEYS:
        raise HTTPException(status_code=403, detail="Invalid API key")
    return creds.credentials

class ChatRequest(BaseModel):
    messages: list[dict]
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=512, ge=1, le=4096)
    top_p: float = Field(default=0.95, ge=0.0, le=1.0)

@app.get("/health")
async def health():
    try:
        async with httpx.AsyncClient(timeout=5.0) as c:
            backend_ok = (await c.get(f"{VLLM_BASE_URL}/models")).status_code == 200
    except Exception:
        backend_ok = False
    return {"status": "healthy" if backend_ok else "degraded",
            "backend": "connected" if backend_ok else "unreachable"}

@app.post("/v1/chat/completions")
@limiter.limit("<YOUR_RATE_LIMIT>")  # e.g., "100/minute"
async def chat(request: Request, body: ChatRequest,
                api_key: str = Depends(verify_api_key)):
    rid = hashlib.md5(f"{time.time()}".encode()).hexdigest()[:12]
    logger.info(f"[{rid}] messages={len(body.messages)}")
    payload = {"model": "<YOUR_MODEL_NAME>", "messages": body.messages,
               "temperature": body.temperature, "max_tokens": body.max_tokens,
               "top_p": body.top_p}
    try:
        async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT) as c:
            resp = await c.post(f"{VLLM_BASE_URL}/chat/completions", json=payload)
            resp.raise_for_status()
            data = resp.json()
        return {"id": rid, "object": "chat.completion", "model": "<YOUR_MODEL_NAME>",
                "choices": data["choices"], "usage": data.get("usage", {})}
    except httpx.TimeoutException:
        raise HTTPException(status_code=504, detail="Backend timeout")
    except Exception as e:
        raise HTTPException(status_code=502, detail=str(e)[:200])

@app.get("/metrics")
async def metrics(api_key: str = Depends(verify_api_key)):
    return {"active_keys": len(API_KEYS), "backend": VLLM_BASE_URL}
'''

gw_dir = PROJECT_ROOT / "gateway"
with open(gw_dir / "main.py", "w") as f: f.write(GATEWAY_CODE.strip())
with open(gw_dir / "requirements.txt", "w") as f:
    f.write("fastapi==0.115.0\nuvicorn[standard]==0.30.0\nslowapi==0.1.9\n"
            "python-dotenv==1.0.0\nhttpx==0.27.0\npydantic>=2.0.0\n")
with open(gw_dir / ".env.example", "w") as f:
    f.write("VLLM_BASE_URL=http://localhost:8000/v1\nAPI_KEYS=<YOUR_KEY_1>,<YOUR_KEY_2>\nREQUEST_TIMEOUT=120\n")

print("Gateway files created: gateway/main.py, gateway/requirements.txt, gateway/.env.example")
print("Launch: cd gateway && uvicorn main:app --host 0.0.0.0 --port 8080")

---
## Part 7: Docker Containerization

Multi-service deployment with Docker Compose.

In [ ]:
# ---- Part 7: Dockerfiles & docker-compose.yml ----

dk = PROJECT_ROOT / "docker"

with open(dk / "Dockerfile.gateway", "w") as f:
    f.write(r'''FROM python:3.11-slim
WORKDIR /app
RUN groupadd -r appuser && useradd -r -g appuser appuser
COPY gateway/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY gateway/ ./
USER appuser
EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \
    CMD python -c "import httpx; httpx.get('http://localhost:8080/health').raise_for_status()"
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
''')

with open(dk / "Dockerfile.vllm", "w") as f:
    f.write(r'''FROM vllm/vllm-openai:latest
ENV VLLM_MODEL_PATH=<YOUR_AWQ_MODEL_PATH>
ENV VLLM_MODEL_NAME=<YOUR_MODEL_NAME>
EXPOSE 8000
CMD ["--model", "${VLLM_MODEL_PATH}", "--host", "0.0.0.0", "--port", "8000",
     "--quantization", "awq", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90", "--served-model-name", "${VLLM_MODEL_NAME}"]
''')

with open(PROJECT_ROOT / "docker-compose.yml", "w") as f:
    f.write(r'''version: "3.8"
services:
  vllm:
    build: { context: ., dockerfile: docker/Dockerfile.vllm }
    image: <YOUR_REGISTRY>/vllm-<YOUR_APP>:latest
    container_name: vllm-backend
    ports: ["8000:8000"]
    volumes: [<YOUR_LOCAL_MODEL_DIR>:/models:ro]
    environment:
      - VLLM_MODEL_PATH=/models/awq-quantized
      - VLLM_MODEL_NAME=<YOUR_MODEL_NAME>
      - CUDA_VISIBLE_DEVICES=0
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia; count: 1; capabilities: [gpu]
    restart: unless-stopped

  gateway:
    build: { context: ., dockerfile: docker/Dockerfile.gateway }
    image: <YOUR_REGISTRY>/gateway-<YOUR_APP>:latest
    container_name: api-gateway
    ports: ["8080:8080"]
    environment:
      - VLLM_BASE_URL=http://vllm:8000/v1
      - API_KEYS=${API_KEYS}
      - REQUEST_TIMEOUT=120
    depends_on: { vllm: { condition: service_healthy } }
    restart: unless-stopped
''')

print("Docker files created: docker/Dockerfile.gateway, docker/Dockerfile.vllm, docker-compose.yml")
print("\nBuild & run:")
print("  docker compose build")
print("  API_KEYS='<YOUR_KEYS>' docker compose up -d")

---
## Part 8: Automated Evaluation — Base vs Fine-Tuned

Compare models using BLEU-4 and ROUGE-L on a held-out test set.

In [ ]:
# ---- Part 8: Evaluation Setup & Metrics ----
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def compute_bleu(ref, cand):
    return sentence_bleu([nltk.word_tokenize(ref.lower())], nltk.word_tokenize(cand.lower()),
                         smoothing_function=SmoothingFunction().method4)

def compute_rouge_l(ref, cand):
    return rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True).score(ref, cand)["rougeL"].fmeasure

def evaluate_model(model_name, test_data, generate_fn):
    results = []
    for i, item in enumerate(test_data):
        prompt = (f"<|im_start|>user\n{item['instruction']}\n<|im_end|>\n<|im_start|>assistant\n")
        candidate = generate_fn(prompt)
        ref = item["reference"]
        results.append({"id": i, "model": model_name,
                         "BLEU-4": round(compute_bleu(ref, candidate), 4),
                         "ROUGE-L": round(compute_rouge_l(ref, candidate), 4)})
    return pd.DataFrame(results)

# Test set — replace with domain-specific Q&A
TEST_QUESTIONS = [
    {"instruction": "<YOUR_TEST_Q_1>", "reference": "<YOUR_GOLD_ANSWER_1>"},
    {"instruction": "<YOUR_TEST_Q_2>", "reference": "<YOUR_GOLD_ANSWER_2>"},
    {"instruction": "<YOUR_TEST_Q_3>", "reference": "<YOUR_GOLD_ANSWER_3>"},
    {"instruction": "<YOUR_TEST_Q_4>", "reference": "<YOUR_GOLD_ANSWER_4>"},
    {"instruction": "<YOUR_TEST_Q_5>", "reference": "<YOUR_GOLD_ANSWER_5>"},
]

Dataset.from_list(TEST_QUESTIONS).to_json(str(DATA_DIR / "test_set.jsonl"))
print(f"Test set: {len(TEST_QUESTIONS)} questions")
print("\nTo run real evaluation, implement generate_fn for each model:")
print("  - vLLM OpenAI client or vLLM offline generate")
print("  - transformers model.generate()")

In [ ]:
# ---- Part 8b: Simulated Evaluation Results & Visualization ----

# Replace with real evaluation results
sim_results = pd.DataFrame({
    "Model": ["Base Model"] * 5 + ["QLoRA Fine-Tuned"] * 5 + ["QLoRA + DPO"] * 5,
    "BLEU-4": [0.12, 0.15, 0.10, 0.14, 0.11, 0.22, 0.25, 0.19, 0.24, 0.21, 0.26, 0.28, 0.23, 0.27, 0.24],
    "ROUGE-L": [0.18, 0.21, 0.16, 0.20, 0.17, 0.28, 0.31, 0.25, 0.30, 0.27, 0.32, 0.34, 0.29, 0.33, 0.30],
})

agg = sim_results.groupby("Model").agg(
    BLEU_mean=("BLEU-4", "mean"), ROUGE_mean=("ROUGE-L", "mean")
).reset_index()
print("Aggregate Results:")
print(agg.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, palette in [(axes[0], "BLEU-4", "Blues"), (axes[1], "ROUGE-L", "Oranges")]:
    sns.boxplot(data=sim_results, x="Model", y=metric, palette=palette, ax=ax)
    sns.stripplot(data=sim_results, x="Model", y=metric, color="black", size=5, alpha=0.6, ax=ax)
    ax.set_title(f"{metric} Score by Model", fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Base vs Fine-Tuned vs DPO-Aligned", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

base_bleu = agg[agg["Model"] == "Base Model"]["BLEU_mean"].values[0]
dpo_bleu = agg[agg["Model"] == "QLoRA + DPO"]["BLEU_mean"].values[0]
print(f"\nBLEU improvement: {base_bleu:.3f} -> {dpo_bleu:.3f} (+{(dpo_bleu-base_bleu)/base_bleu*100:.0f}%)")

---
## Part 9: Production Deployment Documentation

Templates for DEPLOYMENT.md, API.md, and TROUBLESHOOTING.md.

In [ ]:
# ---- Part 9: Generate Documentation ----

docs_dir = PROJECT_ROOT / "docs"

deployment_md = """# Deployment Guide — <YOUR_PROJECT_NAME>
## Prerequisites
GPU Server (RTX 3090/4090 24GB+), Ubuntu 22.04, CUDA 12.1+, Docker 24+ with NVIDIA Container Toolkit
## Quick Start
```bash
git clone <YOUR_REPO_URL> && cd <YOUR_PROJECT_DIR>
cp gateway/.env.example gateway/.env  # edit API_KEYS
docker compose build
API_KEYS='<YOUR_KEYS>' docker compose up -d
curl http://localhost:8080/health
```
## Manual Deployment
```bash
# Terminal 1: vLLM
python -m vllm.entrypoints.openai.api_server --model <YOUR_MODEL> --quantization awq --port 8000
# Terminal 2: Gateway
cd gateway && uvicorn main:app --host 0.0.0.0 --port 8080
```
## Environment Variables
| Variable | Default | Description |
|----------|---------|-------------|
| VLLM_BASE_URL | http://localhost:8000/v1 | vLLM server URL |
| API_KEYS | (required) | Comma-separated valid keys |
| REQUEST_TIMEOUT | 120 | Backend timeout (seconds) |
"""

api_md = """# API Reference — <YOUR_PROJECT_NAME>
## Base URL: `http://<YOUR_SERVER>:8080`
## Auth: `Authorization: Bearer <YOUR_API_KEY>`
## GET /health — Health check (no auth)
## POST /v1/chat/completions — Chat completion (auth required)
```json
{
  "messages": [{"role": "user", "content": "<YOUR_PROMPT>"}],
  "temperature": 0.7, "max_tokens": 512
}
```
## GET /metrics — Service metrics (auth required)
## Error Codes: 403=Invalid key, 429=Rate limited, 502=Backend error, 504=Timeout
"""

troubleshooting_md = """# Troubleshooting — <YOUR_PROJECT_NAME>
## OOM Errors
1. Reduce gpu_memory_utilization (try 0.80)  2. Reduce max_model_len  3. Check nvidia-smi
## 502 Bad Gateway
Check vLLM: `curl http://localhost:8000/health`. Verify VLLM_BASE_URL in .env
## 403 Forbidden
Verify API key is in API_KEYS env var
## Slow Inference
1. Enable prefix caching  2. Increase max_num_seqs  3. Install Flash Attention 2
## NaN Loss During Training
1. Reduce learning rate  2. Add gradient clipping  3. Check data quality
## Docker: GPU not found
```bash
sudo apt-get install -y nvidia-container-toolkit && sudo systemctl restart docker
```
"""

with open(docs_dir / "DEPLOYMENT.md", "w") as f: f.write(deployment_md.strip())
with open(docs_dir / "API.md", "w") as f: f.write(api_md.strip())
with open(docs_dir / "TROUBLESHOOTING.md", "w") as f: f.write(troubleshooting_md.strip())

print("Documentation created: docs/DEPLOYMENT.md, docs/API.md, docs/TROUBLESHOOTING.md")

---
## Final Delivery Checklist

Use this checklist to verify every component of your capstone project before delivery.

### Dataset
| # | Check | Status |
|---|-------|--------|
| 1 | Domain-specific instruction dataset created (100+ examples) | [ ] |
| 2 | Data validation passed (no empty fields, no unfilled placeholders) | [ ] |
| 3 | Train/validation split created (80/20) | [ ] |
| 4 | Chat template formatting verified for target model | [ ] |
| 5 | Preference pairs created for DPO (10+ pairs) | [ ] |
| 6 | Held-out test set saved (5+ questions with gold references) | [ ] |

### Training
| # | Check | Status |
|---|-------|--------|
| 7 | QLoRA SFT completed and loss decreased | [ ] |
| 8 | LoRA adapter weights saved to models/qlora-adapter/ | [ ] |
| 9 | Training loss curves logged (wandb/tensorboard) | [ ] |
| 10 | DPO alignment completed (beta tuned) | [ ] |
| 11 | Final merged model saved in FP16 | [ ] |

### Quantization
| # | Check | Status |
|---|-------|--------|
| 12 | AWQ quantization completed without errors | [ ] |
| 13 | Quantized model loads in vLLM | [ ] |
| 14 | Model size comparison documented (~3.5x compression) | [ ] |
| 15 | Spot-check: 3 test prompts produce reasonable outputs | [ ] |

### Serving
| # | Check | Status |
|---|-------|--------|
| 16 | vLLM server starts and responds to /v1/models | [ ] |
| 17 | OpenAI-compatible endpoint verified | [ ] |
| 18 | Inference latency within SLA (<10s for 256 tokens) | [ ] |

### Gateway
| # | Check | Status |
|---|-------|--------|
| 19 | FastAPI gateway starts without errors | [ ] |
| 20 | API key auth: valid keys accepted, invalid keys rejected | [ ] |
| 21 | Rate limiting active (429 after exceeding limit) | [ ] |
| 22 | Health check returns correct backend status | [ ] |
| 23 | OpenAPI docs accessible at `/docs` | [ ] |

### Containerization
| # | Check | Status |
|---|-------|--------|
| 24 | Dockerfile.gateway builds successfully | [ ] |
| 25 | docker compose up starts both services | [ ] |
| 26 | GPU accessible inside vLLM container | [ ] |
| 27 | Health checks pass for both containers | [ ] |

### Evaluation
| # | Check | Status |
|---|-------|--------|
| 28 | Evaluation script runs end-to-end | [ ] |
| 29 | BLEU and ROUGE-L scores computed | [ ] |
| 30 | Base vs fine-tuned comparison visualized | [ ] |
| 31 | Improvement quantified (% gain over baseline) | [ ] |

### Documentation
| # | Check | Status |
|---|-------|--------|
| 32 | DEPLOYMENT.md complete with all sections | [ ] |
| 33 | API.md complete with endpoint documentation | [ ] |
| 34 | TROUBLESHOOTING.md covers common issues | [ ] |
| 35 | README.md has project overview and quick start | [ ] |
| 36 | All `<YOUR_...>` placeholders replaced with actual values | [ ] |

### Code Quality & Production Readiness
| # | Check | Status |
|---|-------|--------|
| 37 | No hardcoded secrets (keys in env vars only) | [ ] |
| 38 | requirements.txt up to date | [ ] |
| 39 | .gitignore excludes model weights, venv, secrets | [ ] |
| 40 | Error handling with meaningful messages | [ ] |
| 41 | Logging configured with appropriate levels | [ ] |
| 42 | Graceful shutdown handling implemented | [ ] |
| 43 | Resource limits configured (Docker memory/CPU) | [ ] |
| 44 | Backup strategy for model weights documented | [ ] |
| 45 | Repository structure matches the directory tree | [ ] |

---
## Summary

### What You Built

| Stage | Component | Key Technology |
|-------|-----------|---------------|
| 1. Data | Instruction dataset + preference pairs | Template-based generation, ChatML formatting |
| 2. SFT | QLoRA fine-tuning | 4-bit NF4 + LoRA (r=16), SFTTrainer |
| 3. Alignment | DPO preference optimization | DPOTrainer, beta=0.1 |
| 4. Quantization | AWQ 4-bit compression | autoawq, ~3.5x size reduction |
| 5. Serving | High-throughput inference | vLLM, PagedAttention, continuous batching |
| 6. Gateway | Auth + rate limiting | FastAPI, slowapi, Bearer token auth |
| 7. Deployment | Containerization | Docker multi-stage build, docker compose |
| 8. Evaluation | Automated metrics | BLEU-4, ROUGE-L, pandas + matplotlib |
| 9. Docs | Production documentation | DEPLOYMENT.md, API.md, TROUBLESHOOTING.md |

### All 4 Stages of the LLM Learning Roadmap — Complete

1. **Foundations** — Transformer architecture, tokenization, attention mechanisms
2. **Training & Fine-Tuning** — Full fine-tuning, LoRA, QLoRA, instruction tuning
3. **Optimization** — Quantization (AWQ/GPTQ), inference optimization, VRAM management
4. **Production** — Distributed training (DeepSpeed/FSDP), troubleshooting, deployment

### Next Steps
- Deploy to a cloud GPU instance (AWS g5, Lambda Labs, RunPod)
- Set up CI/CD for automated testing and deployment
- Add a web UI (Gradio, Streamlit, or custom frontend)
- Implement RAG (retrieval-augmented generation) for knowledge grounding
- Add monitoring (Prometheus + Grafana) for production observability
- Scale to multi-GPU with tensor parallelism